## Librerias

In [19]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

## Generación de individuos y fitness

In [20]:
n = 8   # individuos
m = 10  # cromosomas

poblacion = np.random.randint(0, 2, (n, m))

def fitness(individuo):
    return np.sum(individuo)

fitness_poblacion = np.array([fitness(i) for i in poblacion])

print("Población:")
print(poblacion)
print("\nFitness:")
print(fitness_poblacion)

Población:
[[0 1 0 1 1 0 1 1 1 1]
 [1 1 1 0 1 1 0 1 0 1]
 [1 0 0 0 0 1 1 1 0 0]
 [0 1 1 1 0 1 1 0 0 0]
 [0 1 1 0 1 1 1 1 1 1]
 [1 1 1 1 1 0 0 1 0 0]
 [1 1 1 0 1 1 0 0 1 1]
 [0 0 1 0 0 1 1 0 1 0]]

Fitness:
[7 7 4 5 8 6 7 4]


## Dos metodos

### Selección por ruleta

In [21]:
def seleccion_ruleta(poblacion, fitness):
    probabilidades = fitness / fitness.sum()
    i = np.random.choice(len(poblacion), p=probabilidades)
    return poblacion[i]

# Elegir 2 padres diferentes para visualizar la diferencia en los cruzamientos
padre1 = seleccion_ruleta(poblacion, fitness_poblacion)
padre2 = poblacion[np.argmin(fitness_poblacion)] if np.array_equal(padre1, poblacion[0]) else poblacion[0]

print("Padre 1:", padre1)
print("Padre 2:", padre2)

Padre 1: [0 1 0 1 1 0 1 1 1 1]
Padre 2: [1 0 0 0 0 1 1 1 0 0]


### Selección por torneo

In [22]:
def seleccion_torneo(poblacion, fitness, k=3):
    indices = np.random.choice(len(poblacion), k, replace=False)
    ganador = indices[np.argmax(fitness[indices])]
    return poblacion[ganador]

padre1 = seleccion_torneo(poblacion, fitness_poblacion)
# Garantizar que Padre 2 sea un individuo distinto de Padre 1
i1 = np.argmax(fitness_poblacion)
i2 = (i1 + 1) % len(poblacion)
padre1 = poblacion[i1]
padre2 = poblacion[i2]

print("Padre 1:", padre1)
print("Padre 2:", padre2)

Padre 1: [0 1 1 0 1 1 1 1 1 1]
Padre 2: [1 1 1 1 1 0 0 1 0 0]


## Metodo de Cruzamientos

### Cruzamiento de 1 punto

In [23]:
def cruce_1_punto(p1, p2):
    punto = np.random.randint(1, len(p1))
    h1 = np.r_[p1[:punto], p2[punto:]]
    h2 = np.r_[p2[:punto], p1[punto:]]
    return h1, h2

h1, h2 = cruce_1_punto(padre1, padre2)

print("Hijo 1:", h1)
print("Hijo 2:", h2)

Hijo 1: [0 1 1 0 1 1 1 1 1 0]
Hijo 2: [1 1 1 1 1 0 0 1 0 1]


### Cruzamiento de 2 puntos

In [24]:
def cruce_2_puntos(p1, p2):
    a, b = sorted(np.random.choice(range(1, len(p1)), 2, replace=False))
    h1 = p1.copy()
    h2 = p2.copy()
    h1[a:b], h2[a:b] = p2[a:b], p1[a:b]
    return h1, h2

h1, h2 = cruce_2_puntos(padre1, padre2)

print("Hijo 1:", h1)
print("Hijo 2:", h2)

Hijo 1: [0 1 1 0 1 0 0 1 0 1]
Hijo 2: [1 1 1 1 1 1 1 1 1 0]


### Cruzamiento uniforme

In [25]:
def cruce_uniforme(p1, p2):
    mascara = np.random.randint(0, 2, len(p1))
    h1 = np.where(mascara, p1, p2)
    h2 = np.where(mascara, p2, p1)
    return h1, h2

h1, h2 = cruce_uniforme(padre1, padre2)

print("Hijo 1:", h1)
print("Hijo 2:", h2)

Hijo 1: [0 1 1 0 1 1 0 1 0 1]
Hijo 2: [1 1 1 1 1 0 1 1 1 0]


## gráficos dinámico de la selección y los métodos de cruzamiento.

### Gráfico dinámico de selección

In [26]:
def graficar_seleccion(metodo):
    plt.figure()
    if metodo == "Ruleta":
        seleccionado = seleccion_ruleta(poblacion, fitness_poblacion)
    else:
        seleccionado = seleccion_torneo(poblacion, fitness_poblacion)

    plt.bar(range(n), fitness_poblacion)
    plt.axhline(fitness(seleccionado), linestyle="--")
    plt.xlabel("Individuos")
    plt.ylabel("Fitness")
    plt.title("Selección por " + metodo)
    plt.show()

widgets.interact(
    graficar_seleccion,
    metodo=["Ruleta", "Torneo"]
);

interactive(children=(Dropdown(description='metodo', options=('Ruleta', 'Torneo'), value='Ruleta'), Output()),…

### Gráfico dinámico de cruzamiento

In [27]:
def graficar_cruzamiento(metodo):
    plt.figure()
    if metodo == "1 Punto":
        h1, h2 = cruce_1_punto(padre1, padre2)
    elif metodo == "2 Puntos":
        h1, h2 = cruce_2_puntos(padre1, padre2)
    else:
        h1, h2 = cruce_uniforme(padre1, padre2)

    matriz = np.array([padre1, padre2, h1, h2])
    plt.imshow(matriz, cmap="binary")
    plt.yticks([0, 1, 2, 3], ["Padre 1", "Padre 2", "Hijo 1", "Hijo 2"])
    plt.xlabel("Genes")
    plt.title("Cruzamiento por " + metodo)
    plt.show()

widgets.interact(
    graficar_cruzamiento,
    metodo=["1 Punto", "2 Puntos", "Uniforme"]
);

interactive(children=(Dropdown(description='metodo', options=('1 Punto', '2 Puntos', 'Uniforme'), value='1 Pun…